# Chapter 68: Compliance Automation

**Volume 4 — Security Operations**

**The audit scramble ends when compliance becomes continuous.**
Two months of pre-audit panic, manual screenshots, and Excel-based evidence collection
is not a compliance program — it is compliance theater.
AI automation turns compliance from a periodic destination into a continuous background process.
The audit becomes a **report generation exercise**, not an evidence excavation.

### What You Will Build — 5 Demos

| Demo | Engine | What It Does |
|------|--------|--------------|
| **1. Config Compliance Checker** | Deterministic rules + LLM | Network device configs vs CIS/NIST/PCI baseline |
| **2. Access Rights Auditor** | Policy engine | Orphaned accounts, excessive privilege, violations |
| **3. Change Compliance Scorer** | Impact analyzer | Did this change violate a control? Severity score |
| **4. Evidence Collector** | Snapshot engine | Automated audit package generation with timestamps |
| **5. Compliance Gap Reporter** | LLM Sonnet | Maps findings to PCI-DSS, SOC 2, NIST CSF controls |

### The Core Insight
> **Compliance is not a state you reach and maintain — it is a state that degrades
> continuously unless you actively fight the entropy.
> Every configuration change introduces potential drift. AI-powered continuous monitoring
> is the OSPF hello packet for your compliance LSDB:
> it detects drift the moment it happens, not two months later during the audit.**

In [ ]:
# pip install anthropic
import os, json, re, math, time, random
from dataclasses import dataclass, field
from typing import List, Dict, Optional
from collections import defaultdict

try:
    from google.colab import userdata
    os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')
except Exception:
    import getpass
    if not os.environ.get('ANTHROPIC_API_KEY'):
        os.environ['ANTHROPIC_API_KEY'] = getpass.getpass('Enter your Anthropic API key: ')

api_key = os.environ.get('ANTHROPIC_API_KEY', '')
USE_API = bool(api_key)

if USE_API:
    from anthropic import Anthropic
    client = Anthropic()
    print("LLM analyst: ACTIVE (Anthropic API)")
else:
    print("LLM analyst: SIMULATION MODE (set ANTHROPIC_API_KEY for real analysis)")

def llm_analyze(prompt: str, max_tokens: int = 200) -> str:
    if USE_API:
        resp = client.messages.create(
            model="claude-haiku-4-5-20251001",
            max_tokens=max_tokens,
            messages=[{"role": "user", "content": prompt}]
        )
        return resp.content[0].text.strip()
    return "Simulation: " + prompt[:80] + "..."

print("Setup complete.")

## Demo 1: Configuration Compliance Checker

Network device configurations are among the highest-value compliance artifacts.
PCI-DSS Requirements 1 and 2 are almost entirely about **network configuration**.

This demo checks a Cisco IOS firewall against a multi-framework baseline:

| Control | Framework | Check Type | What We Test |
|---------|-----------|------------|-------------|
| Req 2.2.7 | PCI-DSS | Deterministic | No telnet, SSH v2 only |
| Req 8.3.6 | PCI-DSS | Deterministic | Password min-length >= 12 |
| Req 10.2.1 | PCI-DSS | Deterministic | Remote syslog configured |
| PR.AC-7 | NIST CSF | Deterministic | MFA/strong auth enforced |
| Req 1.3.1 | PCI-DSS | Semantic (LLM) | CDE segmentation adequate? |
| CC6.1 | SOC 2 | Semantic (LLM) | Access controls principle of least privilege? |

**Two-layer evaluation** — the same approach used in production compliance platforms:
- **Deterministic**: binary pass/fail from config pattern matching
- **Semantic (LLM)**: interpreted assessment for controls requiring judgment

*Network analogy: this is your config compliance check in Cisco DNA Center or Ansible
— but instead of just checking if NTP is configured, it also understands whether
the ACL structure actually achieves the segmentation objective.*

In [ ]:
# ── Demo 1: Configuration Compliance Checker ───────────────────────────────────

@dataclass
class ComplianceCheck:
    control_id:   str
    framework:    str
    requirement:  str
    check_type:   str     # deterministic | semantic
    result:       str     # PASS | FAIL | NEEDS_REVIEW
    evidence:     str
    severity:     str     # critical | high | medium | low


# ── Sample Cisco IOS device configs (intentionally mixed: some compliant, some not) ───
DEVICE_CONFIGS = {
    "core-fw-01": """
hostname core-fw-01
!
service ssh version 2
no service telnet
ip ssh time-out 60
ip ssh authentication-retries 3
!
security passwords min-length 12
login block-for 300 attempts 5 within 60
!
logging host 10.0.10.5
logging trap informational
logging buffered 64000
!
ntp server 10.0.0.1
!
aaa new-model
aaa authentication login default group tacacs+ local
!
! CDE segmentation: cardholder environment is 10.1.100.0/24
ip access-list extended OUTSIDE_IN
 permit tcp any 10.1.100.0 0.0.0.255 eq 443
 deny   ip any 10.1.100.0 0.0.0.255
 permit ip any any
!
banner login ^UNAUTHORIZED ACCESS PROHIBITED. ALL ACTIVITY LOGGED.^
""",
    "edge-sw-02": """
hostname edge-sw-02
!
! WARNING: telnet still enabled
service telnet
! no ssh version forced
!
! No password minimum length set
!
! No remote logging
logging buffered 8192
!
! No NTP
! No AAA
!
! No CDE ACL
ip access-list extended VLAN10_IN
 permit ip any any
""",
}


def run_deterministic_checks(device_name: str, config: str) -> List[ComplianceCheck]:
    """Binary pattern-matching checks — no interpretation needed."""
    checks = []

    # PCI-DSS 2.2.7: No telnet, SSH v2 only
    has_no_telnet  = "no service telnet" in config
    has_ssh_v2     = "service ssh version 2" in config
    checks.append(ComplianceCheck(
        control_id="PCI-DSS-2.2.7", framework="PCI-DSS",
        requirement="All non-console administrative access must be encrypted (no telnet)",
        check_type="deterministic",
        result="PASS" if (has_no_telnet and has_ssh_v2) else "FAIL",
        evidence=(
            "Telnet disabled, SSH v2 configured"
            if (has_no_telnet and has_ssh_v2)
            else f"FAIL: telnet={'disabled' if has_no_telnet else 'ENABLED'}, "
                 f"ssh_v2={'yes' if has_ssh_v2 else 'NO'}"
        ),
        severity="critical"
    ))

    # PCI-DSS 8.3.6: Password minimum length >= 12
    min_len_match = re.search(r"security passwords min-length (\d+)", config)
    min_len = int(min_len_match.group(1)) if min_len_match else 0
    checks.append(ComplianceCheck(
        control_id="PCI-DSS-8.3.6", framework="PCI-DSS",
        requirement="Passwords must meet minimum length of 12 characters",
        check_type="deterministic",
        result="PASS" if min_len >= 12 else "FAIL",
        evidence=(
            f"min-length = {min_len}" if min_len >= 12
            else f"FAIL: min-length = {min_len} (need >= 12)"
        ),
        severity="high"
    ))

    # PCI-DSS 10.2.1: Centralized syslog
    has_remote_log = bool(re.search(r"logging host \d+\.\d+\.\d+\.\d+", config))
    checks.append(ComplianceCheck(
        control_id="PCI-DSS-10.2.1", framework="PCI-DSS",
        requirement="Audit logs forwarded to a centralized, secure log server",
        check_type="deterministic",
        result="PASS" if has_remote_log else "FAIL",
        evidence="Remote syslog server configured" if has_remote_log else "FAIL: No 'logging host' configured",
        severity="high"
    ))

    # NIST CSF PR.AC-7: Strong authentication (AAA)
    has_aaa = "aaa new-model" in config
    has_tacacs = "group tacacs+" in config or "group radius" in config
    checks.append(ComplianceCheck(
        control_id="NIST-PR.AC-7", framework="NIST CSF",
        requirement="Users and devices authenticated commensurate with risk",
        check_type="deterministic",
        result="PASS" if (has_aaa and has_tacacs) else ("PASS" if has_aaa else "FAIL"),
        evidence=(
            f"AAA enabled, {'TACACS+' if has_tacacs else 'local fallback only'}"
            if has_aaa else "FAIL: No AAA model configured"
        ),
        severity="critical"
    ))

    return checks


def run_semantic_checks(device_name: str, config: str) -> List[ComplianceCheck]:
    """LLM-based checks for controls requiring interpretation."""
    prompt = (
        f"You are a PCI-DSS QSA and SOC 2 auditor assessing a Cisco IOS configuration.\n"
        f"Device: {device_name}\n\n"
        f"Configuration:\n```\n{config.strip()}\n```\n\n"
        f"Assess these two controls. Reply in this EXACT format for each:\n"
        f"CONTROL: <id>\nRESULT: <PASS|FAIL|NEEDS_REVIEW>\nEVIDENCE: <one sentence citing config lines>\n\n"
        f"Controls to assess:\n"
        f"1. PCI-DSS-1.3.1: Inbound traffic to the CDE (10.1.100.0/24) is restricted "
        f"to only what is necessary — no unauthorized paths exist.\n"
        f"2. SOC2-CC6.1: Logical access security measures restrict access to "
        f"information assets — principle of least privilege is evident."
    )
    response = llm_analyze(prompt, max_tokens=250)

    # Parse structured response
    checks = []
    blocks = response.split("\n\n")
    expected = [
        ("PCI-DSS-1.3.1", "PCI-DSS",
         "Inbound to CDE restricted to necessary traffic only", "critical"),
        ("SOC2-CC6.1", "SOC 2",
         "Logical access controls enforce least privilege", "high"),
    ]
    for i, block in enumerate(blocks[:2]):
        result_m  = re.search(r"RESULT:\s*(PASS|FAIL|NEEDS_REVIEW)", block, re.IGNORECASE)
        evidence_m = re.search(r"EVIDENCE:\s*(.+)", block)
        ctrl_id, framework, requirement, severity = expected[i] if i < len(expected) else ("UNKNOWN", "", "", "medium")
        checks.append(ComplianceCheck(
            control_id=ctrl_id, framework=framework,
            requirement=requirement,
            check_type="semantic",
            result=result_m.group(1).upper() if result_m else "NEEDS_REVIEW",
            evidence=evidence_m.group(1).strip() if evidence_m else block[:150],
            severity=severity
        ))
    return checks


# ── Run checks on all devices ──────────────────────────────────────────────────
print("=" * 65)
print("CONFIGURATION COMPLIANCE CHECKER — MULTI-FRAMEWORK")
print("=" * 65)

all_device_results: Dict[str, List[ComplianceCheck]] = {}

for device_name, config in DEVICE_CONFIGS.items():
    print(f"\n{'='*60}")
    print(f"Device: {device_name}")
    print(f"{'='*60}")

    det_checks = run_deterministic_checks(device_name, config)
    sem_checks = run_semantic_checks(device_name, config)
    all_checks = det_checks + sem_checks
    all_device_results[device_name] = all_checks

    pass_count = sum(1 for c in all_checks if c.result == "PASS")
    fail_count = sum(1 for c in all_checks if c.result == "FAIL")
    review_count = sum(1 for c in all_checks if c.result == "NEEDS_REVIEW")

    for c in all_checks:
        icon = "PASS" if c.result == "PASS" else ("FAIL" if c.result == "FAIL" else "REVIEW")
        tag  = f"[{c.check_type[:3].upper()}]"
        print(f"  [{icon:6}] {c.control_id:18} {tag:5} {c.severity:8}")
        if c.result != "PASS":
            print(f"           Evidence: {c.evidence}")

    print(f"\n  Score: {pass_count} PASS / {fail_count} FAIL / {review_count} NEEDS_REVIEW")
    if fail_count > 0:
        print(f"  Status: NON-COMPLIANT — {fail_count} control(s) failed")
    else:
        print("  Status: COMPLIANT (subject to review items)")

print("\n[Compliance Checker] Multi-framework assessment complete.")

## Demo 2: Access Rights Auditor

**PCI-DSS Requirement 7** (restrict access to system components to only those individuals
whose job requires such access) and **SOC 2 CC6.3** (logical access restrictions)
both require regular review of who has access to what — and removing access that is
no longer needed.

Manual access review in large environments is miserable:
- Export Active Directory group memberships to Excel
- Compare against HR system for active employees
- Identify accounts not logged in for 90 days
- Check for users with access beyond their role

This demo **automates that review** and uses LLM to explain violations:

| Violation Category | Example | Policy Violated |
|--------------------|---------|----------------|
| **Orphaned account** | account exists, employee terminated | PCI-DSS 8.2.6, SOC2 CC6.3 |
| **Stale account** | no login in > 90 days | PCI-DSS 8.2.7 |
| **Excessive privilege** | developer in Domain Admins | PCI-DSS 7.2.2 |
| **Role violation** | finance user in firewall admins | NIST PR.AC-4 |

*Network analogy: this is your BGP neighbor table review — you know which peers
should be there. Any entry that does not match your approved peer list is a
configuration error (or worse). Same logic: any account that does not match
the approved user roster with approved role is a compliance finding.*

In [ ]:
# ── Demo 2: Access Rights Auditor ─────────────────────────────────────────────

from datetime import datetime, timedelta

@dataclass
class UserAccount:
    username:       str
    display_name:   str
    department:     str
    role:           str
    groups:         List[str]
    last_login:     Optional[str]   # ISO date string or None
    account_enabled: bool
    hr_active:      bool            # Is this person still employed?


@dataclass
class AccessViolation:
    username:    str
    violation:   str        # ORPHANED | STALE | EXCESSIVE_PRIVILEGE | ROLE_VIOLATION
    severity:    str        # critical | high | medium
    detail:      str
    controls:    List[str]  # PCI-DSS / SOC2 / NIST controls violated


# ── Role-to-allowed-groups policy ─────────────────────────────────────────────
ROLE_POLICY: Dict[str, List[str]] = {
    "Developer":         ["VPN_Users", "Dev_Team", "Source_Control"],
    "Finance":           ["VPN_Users", "Finance_Users", "ERP_Access"],
    "IT_Operations":     ["VPN_Users", "IT_Ops", "Network_Admins", "Server_Admins"],
    "Security":          ["VPN_Users", "Security_Team", "SIEM_Access", "Firewall_Admins"],
    "HR":                ["VPN_Users", "HR_Team", "HRIS_Access"],
    "Executive":         ["VPN_Users", "Executives"],
}

HIGH_PRIVILEGE_GROUPS = {"Domain_Admins", "Enterprise_Admins", "Firewall_Admins",
                          "Schema_Admins", "Server_Admins"}


# ── Simulated user account inventory ──────────────────────────────────────────
# In production: pull from Active Directory via ldap3 or from Okta/Azure AD API
TODAY = datetime(2024, 1, 15)

USER_ACCOUNTS = [
    UserAccount("alice",     "Alice Chen",     "Engineering",   "Developer",
                ["VPN_Users", "Dev_Team", "Source_Control"],
                "2024-01-14", True, True),
    UserAccount("bob.smith", "Bob Smith",      "Finance",        "Finance",
                ["VPN_Users", "Finance_Users", "ERP_Access", "Domain_Admins"],  # EXCESSIVE!
                "2024-01-13", True, True),
    UserAccount("carol",     "Carol Davis",    "IT Ops",         "IT_Operations",
                ["VPN_Users", "IT_Ops", "Network_Admins", "Server_Admins"],
                "2024-01-15", True, True),
    UserAccount("dave.old",  "Dave Oldemployee","Engineering",  "Developer",
                ["VPN_Users", "Dev_Team", "Source_Control", "Server_Admins"],  # ORPHANED + EXCESSIVE
                "2023-09-15", True, False),   # NOT in HR = terminated!
    UserAccount("eve",       "Eve Johnson",    "Finance",        "Finance",
                ["VPN_Users", "Finance_Users", "Firewall_Admins"],  # ROLE VIOLATION!
                "2024-01-10", True, True),
    UserAccount("svc_etl",   "ETL Service Acct","IT Ops",       "IT_Operations",
                ["VPN_Users", "IT_Ops"],
                "2023-06-01", True, True),    # STALE (service account, no login in > 90d)
    UserAccount("frank",     "Frank Miller",   "Security",       "Security",
                ["VPN_Users", "Security_Team", "SIEM_Access", "Firewall_Admins"],
                "2024-01-14", True, True),
]


def audit_user(user: UserAccount, stale_days: int = 90) -> List[AccessViolation]:
    """Audit a single user account for policy violations."""
    violations = []

    # 1. Orphaned account: employee no longer in HR system
    if not user.hr_active and user.account_enabled:
        violations.append(AccessViolation(
            username=user.username,
            violation="ORPHANED",
            severity="critical",
            detail=f"{user.display_name} is no longer in HR system but account is still active",
            controls=["PCI-DSS-8.2.6", "SOC2-CC6.3", "NIST-PR.AC-1"]
        ))

    # 2. Stale account: no login in > stale_days
    if user.last_login:
        last = datetime.strptime(user.last_login, "%Y-%m-%d")
        days_inactive = (TODAY - last).days
        if days_inactive > stale_days and user.account_enabled:
            violations.append(AccessViolation(
                username=user.username,
                violation="STALE",
                severity="high",
                detail=f"No login in {days_inactive} days (threshold: {stale_days})",
                controls=["PCI-DSS-8.2.7", "SOC2-CC6.3"]
            ))

    # 3. Excessive privilege: membership in high-privilege groups not allowed by role
    allowed = set(ROLE_POLICY.get(user.role, []))
    user_groups = set(user.groups)
    high_priv = user_groups & HIGH_PRIVILEGE_GROUPS
    if high_priv and not high_priv.issubset(allowed):
        violations.append(AccessViolation(
            username=user.username,
            violation="EXCESSIVE_PRIVILEGE",
            severity="critical",
            detail=f"Member of high-privilege group(s) not allowed for role {user.role}: {high_priv}",
            controls=["PCI-DSS-7.2.2", "SOC2-CC6.3", "NIST-PR.AC-4"]
        ))

    # 4. Role violation: any group not in the allowed-for-role list
    unauthorized = user_groups - allowed - HIGH_PRIVILEGE_GROUPS
    if unauthorized:
        violations.append(AccessViolation(
            username=user.username,
            violation="ROLE_VIOLATION",
            severity="high",
            detail=f"Group memberships outside role policy: {unauthorized}",
            controls=["PCI-DSS-7.2.1", "NIST-PR.AC-4"]
        ))

    return violations


# ── Run access audit ───────────────────────────────────────────────────────────
print("=" * 65)
print("ACCESS RIGHTS AUDITOR — POLICY VIOLATION DETECTION")
print("=" * 65)

all_violations: List[AccessViolation] = []

for user in USER_ACCOUNTS:
    viols = audit_user(user)
    all_violations.extend(viols)
    status = f"{len(viols)} violation(s)" if viols else "CLEAN"
    print(f"  {user.username:12} [{user.role:14}]  {status}")
    for v in viols:
        print(f"    -> [{v.severity:8}] {v.violation}: {v.detail}")

print(f"\n{'='*65}")
print(f"AUDIT SUMMARY: {len(all_violations)} violation(s) across {len(USER_ACCOUNTS)} accounts")
crit = sum(1 for v in all_violations if v.severity == "critical")
high = sum(1 for v in all_violations if v.severity == "high")
print(f"  CRITICAL: {crit}  |  HIGH: {high}")

# LLM generates remediation guidance for critical violations
crit_violations = [v for v in all_violations if v.severity == "critical"]
if crit_violations:
    viol_text = "\n".join(
        f"- {v.username}: {v.violation} — {v.detail} (controls: {v.controls})"
        for v in crit_violations
    )
    guidance = llm_analyze(
        f"Access audit found {len(crit_violations)} CRITICAL violations:\n{viol_text}\n\n"
        f"For each violation: provide the specific remediation step and timeline. "
        f"Under 120 words. Be direct.",
        max_tokens=180
    )
    print(f"\nRemediation Guidance (LLM):\n{guidance}")

## Demo 3: Change Compliance Scorer

**Every configuration change is a potential compliance event.**
A firewall rule added for a legitimate operational reason might inadvertently
open a path that violates PCI-DSS network segmentation.

Traditional change management records *what* changed but not *what compliance impact* it had.
This demo assigns a **compliance impact score** to each configuration change:

| Change Type | Impact Score | Why |
|-------------|-------------|-----|
| Add allow rule to CDE | HIGH | Could violate PCI-DSS 1.3.1 segmentation |
| Enable plaintext protocol | CRITICAL | Violates PCI-DSS 2.2.7, NIST PR.DS-2 |
| Remove logging config | HIGH | Violates PCI-DSS 10.2, SOC2 CC7.2 |
| Change NTP server | LOW | Operational, no direct control impact |
| Disable MFA for admin group | CRITICAL | Violates PCI-DSS 8.4, SOC2 CC6.1 |

The scorer answers: **"Should this change trigger a compliance review
before it is applied to production?"**
High-impact changes get routed to the compliance team automatically.

*Network analogy: this is pre-deployment validation in your CI/CD pipeline
for network configs — Batfish compliance checking before pushing to production.
Every change goes through compliance scoring before it lands on the device.*

In [ ]:
# ── Demo 3: Change Compliance Scorer ──────────────────────────────────────────

@dataclass
class ConfigChange:
    change_id:    str
    device:       str
    change_type:  str
    description:  str
    before:       str      # config snippet before change
    after:        str      # config snippet after change
    submitted_by: str
    timestamp:    str


@dataclass
class ComplianceImpact:
    change_id:       str
    impact_score:    float    # 0.0 = no impact, 1.0 = critical violation
    impact_level:    str      # CRITICAL | HIGH | MEDIUM | LOW | NONE
    controls_at_risk: List[str]
    requires_review:  bool
    rule_findings:    List[str]
    llm_assessment:   str = ""


# ── Compliance impact rules ────────────────────────────────────────────────────
# Each rule: (description, regex_in_after, score_delta, controls_at_risk)
IMPACT_RULES = [
    ("Plaintext protocol enabled",
     r"(?:^|\n)\s*service telnet|(?:^|\n)\s*no service ssh",
     1.0, ["PCI-DSS-2.2.7", "NIST-PR.DS-2"]),

    ("Allow rule added to CDE subnet",
     r"permit.{0,50}10\.1\.100",
     0.8, ["PCI-DSS-1.3.1", "PCI-DSS-1.3.2"]),

    ("Remote logging removed",
     r"no logging host",
     0.7, ["PCI-DSS-10.2.1", "SOC2-CC7.2"]),

    ("AAA or authentication weakened",
     r"no aaa new-model|no aaa authentication",
     0.9, ["PCI-DSS-8.4.2", "SOC2-CC6.1", "NIST-PR.AC-7"]),

    ("Password policy weakened",
     r"security passwords min-length [1-9](?!\d)|no security passwords",
     0.7, ["PCI-DSS-8.3.6", "SOC2-CC6.1"]),

    ("Permit any added (overly permissive)",
     r"permit\s+ip\s+any\s+any|permit\s+tcp\s+any\s+any",
     0.6, ["PCI-DSS-1.2.1", "NIST-PR.AC-5"]),

    ("NTP server changed (low risk)",
     r"ntp server",
     0.1, ["PCI-DSS-10.6.1"]),

    ("Banner login removed",
     r"no banner login",
     0.2, ["PCI-DSS-8.6.1"]),
]


def score_change(change: ConfigChange) -> ComplianceImpact:
    """Score a configuration change for compliance impact."""
    max_score   = 0.0
    all_controls: set = set()
    findings: List[str] = []

    # Also check for regressions: things removed from 'before' that were compliant
    diff_added   = change.after   # new config lines
    diff_removed = change.before  # old config lines

    for rule_desc, pattern, score, controls in IMPACT_RULES:
        # Check if new 'after' config introduces a risky pattern
        if re.search(pattern, diff_added, re.IGNORECASE | re.MULTILINE):
            max_score = max(max_score, score)
            all_controls.update(controls)
            findings.append(f"[+{score:.1f}] {rule_desc}")

    # Score translation
    if max_score >= 0.9:
        level = "CRITICAL"
    elif max_score >= 0.7:
        level = "HIGH"
    elif max_score >= 0.4:
        level = "MEDIUM"
    elif max_score > 0.0:
        level = "LOW"
    else:
        level = "NONE"

    return ComplianceImpact(
        change_id=change.change_id,
        impact_score=max_score,
        impact_level=level,
        controls_at_risk=sorted(all_controls),
        requires_review=max_score >= 0.7,
        rule_findings=findings
    )


# ── Sample change requests ─────────────────────────────────────────────────────
CHANGES = [
    ConfigChange(
        change_id="CR-2024-0891", device="core-fw-01",
        change_type="acl_modification",
        description="Open port 8443 from internet to CDE for new payment API",
        before="deny ip any 10.1.100.0 0.0.0.255",
        after="permit tcp any 10.1.100.0 0.0.0.255 eq 8443\ndeny ip any 10.1.100.0 0.0.0.255",
        submitted_by="network-engineer@corp.com", timestamp="2024-01-15T09:00:00Z"
    ),
    ConfigChange(
        change_id="CR-2024-0892", device="mgmt-sw-03",
        change_type="auth_change",
        description="Emergency fix: AAA server down, reverting to local auth",
        before="aaa new-model\naaa authentication login default group tacacs+ local",
        after="no aaa new-model",
        submitted_by="oncall-engineer@corp.com", timestamp="2024-01-15T03:22:00Z"
    ),
    ConfigChange(
        change_id="CR-2024-0893", device="core-fw-01",
        change_type="ntp_update",
        description="Update NTP server to new internal stratum-2",
        before="ntp server 10.0.0.1",
        after="ntp server 10.0.0.5",
        submitted_by="netops@corp.com", timestamp="2024-01-15T14:00:00Z"
    ),
    ConfigChange(
        change_id="CR-2024-0894", device="edge-sw-02",
        change_type="protocol_enable",
        description="Vendor requested telnet for legacy device management",
        before="no service telnet",
        after="service telnet",
        submitted_by="vendor-support@vendor.com", timestamp="2024-01-15T11:30:00Z"
    ),
]

print("=" * 65)
print("CHANGE COMPLIANCE SCORER — PRE-DEPLOYMENT IMPACT ANALYSIS")
print("=" * 65)

impacts: List[ComplianceImpact] = []

for change in CHANGES:
    impact = score_change(change)
    impacts.append(impact)

    print(f"\n{change.change_id} | {change.device}")
    print(f"  Description : {change.description}")
    print(f"  Submitted by: {change.submitted_by}")
    print(f"  Impact Level: {impact.impact_level}  (score={impact.impact_score:.2f})")
    print(f"  Controls    : {impact.controls_at_risk}")
    if impact.rule_findings:
        print(f"  Findings    : {impact.rule_findings}")
    print(f"  Review Req'd: {'YES — route to compliance team' if impact.requires_review else 'No — proceed with normal approval'}")

    if impact.requires_review:
        assessment = llm_analyze(
            f"Change request {change.change_id}: {change.description}. "
            f"Config change: remove '{change.before}' -> add '{change.after}'. "
            f"Impact level: {impact.impact_level}. Controls at risk: {impact.controls_at_risk}. "
            f"Should this be approved or rejected? What compensating controls would allow approval? "
            f"Under 80 words.",
            max_tokens=120
        )
        impact.llm_assessment = assessment
        print(f"  LLM        : {assessment}")

print(f"\n[Scorer] {sum(1 for i in impacts if i.requires_review)} of {len(CHANGES)} changes require compliance review.")

## Demo 4: Automated Evidence Collector

SOC 2 Type II requires evidence that controls **operated continuously** throughout
the audit period — not just that they were configured correctly on audit day.

Traditional approach: manually run scripts before the audit, screenshot configs,
export data to Excel, organize into folders.
**This takes weeks and produces evidence that an auditor knows was generated for the audit.**

The automated approach: **continuous, timestamped snapshots** stored in an append-only
evidence log throughout the audit period. When the auditor asks
"was MFA enabled on all admin accounts on November 15th?"
the answer is in the evidence log — automatically.

```
Evidence snapshot structure:
  snapshot_id : EVD-2024-0115-001
  timestamp   : 2024-01-15T06:00:00Z  (automated, daily)
  collector   : compliance-bot-01
  device      : core-fw-01
  control     : PCI-DSS-2.2.7
  status      : COMPLIANT
  evidence    : {"ssh_version": "2", "telnet_enabled": false}
  hash        : sha256:a3f1...  (tamper-evident)
```

The hash makes the evidence **tamper-evident** — an auditor can verify that
the snapshot has not been modified since collection.

In [ ]:
# ── Demo 4: Automated Evidence Collector ──────────────────────────────────────

import hashlib
from datetime import datetime

@dataclass
class EvidenceSnapshot:
    """A single tamper-evident compliance evidence record."""
    snapshot_id:  str
    timestamp:    str
    device:       str
    control_id:   str
    framework:    str
    status:       str     # COMPLIANT | NON_COMPLIANT | DEGRADED
    raw_evidence: dict
    collector:    str
    evidence_hash: str = ""

    def __post_init__(self):
        if not self.evidence_hash:
            self.evidence_hash = self._compute_hash()

    def _compute_hash(self) -> str:
        """SHA-256 of core fields — makes snapshot tamper-evident."""
        content = json.dumps({
            "snapshot_id":  self.snapshot_id,
            "timestamp":    self.timestamp,
            "device":       self.device,
            "control_id":   self.control_id,
            "status":       self.status,
            "raw_evidence": self.raw_evidence,
        }, sort_keys=True)
        return "sha256:" + hashlib.sha256(content.encode()).hexdigest()[:16]


class EvidenceStore:
    """
    Append-only evidence log.
    In production: backed by immutable object storage (AWS S3 Object Lock,
    Azure Blob with WORM policies, or a blockchain-anchored audit log).
    """
    def __init__(self):
        self._log: List[EvidenceSnapshot] = []
        self._counter = 0

    def append(self, snapshot: EvidenceSnapshot):
        """Append-only: no modification after write."""
        self._log.append(snapshot)
        self._counter += 1

    def query(self, device: str = None, control_id: str = None,
              status: str = None) -> List[EvidenceSnapshot]:
        """Filter evidence log by device, control, or status."""
        results = self._log
        if device:     results = [s for s in results if s.device == device]
        if control_id: results = [s for s in results if s.control_id == control_id]
        if status:     results = [s for s in results if s.status == status]
        return results

    def generate_audit_package(
        self, framework: str, start_date: str, end_date: str
    ) -> dict:
        """Generate a structured evidence package for a specific framework and date range."""
        relevant = [s for s in self._log if s.framework == framework
                    and start_date <= s.timestamp <= end_date + "Z"]
        by_control: Dict[str, List[EvidenceSnapshot]] = defaultdict(list)
        for snap in relevant:
            by_control[snap.control_id].append(snap)

        package = {
            "framework":    framework,
            "period":       f"{start_date} to {end_date}",
            "generated_at": datetime.utcnow().isoformat() + "Z",
            "controls":     {}
        }
        for ctrl, snaps in by_control.items():
            compliant_count = sum(1 for s in snaps if s.status == "COMPLIANT")
            total = len(snaps)
            package["controls"][ctrl] = {
                "total_snapshots":     total,
                "compliant":           compliant_count,
                "compliance_rate_pct": round(100 * compliant_count / total, 1) if total else 0,
                "latest_status":       snaps[-1].status,
                "sample_evidence":     snaps[-1].raw_evidence,
                "latest_hash":         snaps[-1].evidence_hash,
            }
        return package


def collect_device_evidence(
    store: EvidenceStore, device: str, config: str,
    timestamp: str, framework: str = "PCI-DSS"
):
    """Extract compliance evidence from a device config and store snapshots."""
    store._counter += 1
    snap_base_id = f"EVD-{timestamp[:10].replace('-','')}-{store._counter:03d}"

    # Evidence extraction: SSH
    store.append(EvidenceSnapshot(
        snapshot_id=snap_base_id + "-ssh",
        timestamp=timestamp, device=device,
        control_id="PCI-DSS-2.2.7", framework=framework,
        status="COMPLIANT" if ("service ssh version 2" in config and "no service telnet" in config)
               else "NON_COMPLIANT",
        raw_evidence={
            "ssh_v2":     "service ssh version 2" in config,
            "telnet_off": "no service telnet" in config
        },
        collector="compliance-automation-bot"
    ))

    # Evidence extraction: Logging
    log_match = re.search(r"logging host ([\d.]+)", config)
    store.append(EvidenceSnapshot(
        snapshot_id=snap_base_id + "-log",
        timestamp=timestamp, device=device,
        control_id="PCI-DSS-10.2.1", framework=framework,
        status="COMPLIANT" if log_match else "NON_COMPLIANT",
        raw_evidence={"logging_host": log_match.group(1) if log_match else None},
        collector="compliance-automation-bot"
    ))

    # Evidence extraction: AAA
    store.append(EvidenceSnapshot(
        snapshot_id=snap_base_id + "-aaa",
        timestamp=timestamp, device=device,
        control_id="PCI-DSS-8.4.2", framework=framework,
        status="COMPLIANT" if "aaa new-model" in config else "NON_COMPLIANT",
        raw_evidence={"aaa_enabled": "aaa new-model" in config,
                      "tacacs": "group tacacs+" in config},
        collector="compliance-automation-bot"
    ))


# ── Simulate 3 daily collection runs ──────────────────────────────────────────
print("=" * 65)
print("AUTOMATED EVIDENCE COLLECTOR — 3-DAY SIMULATION")
print("=" * 65)

store = EvidenceStore()

collection_schedule = [
    ("2024-01-13T06:00:00Z", "core-fw-01",  DEVICE_CONFIGS["core-fw-01"]),
    ("2024-01-13T06:00:00Z", "edge-sw-02",  DEVICE_CONFIGS["edge-sw-02"]),
    ("2024-01-14T06:00:00Z", "core-fw-01",  DEVICE_CONFIGS["core-fw-01"]),
    ("2024-01-14T06:00:00Z", "edge-sw-02",  DEVICE_CONFIGS["edge-sw-02"]),
    # Jan 15: edge-sw-02 gets telnet re-enabled (from CR-2024-0894)
    ("2024-01-15T06:00:00Z", "core-fw-01",  DEVICE_CONFIGS["core-fw-01"]),
    ("2024-01-15T06:00:00Z", "edge-sw-02",  DEVICE_CONFIGS["edge-sw-02"].replace("service telnet", "service telnet  !ENABLED BY CR-2024-0894")),
]

for ts, device, config in collection_schedule:
    collect_device_evidence(store, device, config, ts)

print(f"Collected {len(store._log)} evidence snapshots across 3 days")

# Show non-compliant snapshots
non_compliant = store.query(status="NON_COMPLIANT")
print(f"\nNon-compliant snapshots: {len(non_compliant)}")
for snap in non_compliant:
    print(f"  {snap.timestamp[:10]} | {snap.device:12} | {snap.control_id:18} | {snap.status}")
    print(f"    Evidence: {snap.raw_evidence}")
    print(f"    Hash    : {snap.evidence_hash}")

# Generate audit package
print("\n" + "=" * 65)
print("AUDIT EVIDENCE PACKAGE — PCI-DSS (Jan 13-15)")
print("=" * 65)
package = store.generate_audit_package("PCI-DSS", "2024-01-13", "2024-01-15")
for ctrl, data in package["controls"].items():
    status_icon = "PASS" if data["compliance_rate_pct"] == 100 else "WARN"
    print(f"  [{status_icon}] {ctrl:18}  "
          f"Compliance: {data['compliance_rate_pct']:5.1f}%  "
          f"({data['compliant']}/{data['total_snapshots']} snapshots)  "
          f"Latest: {data['latest_status']}")

print(f"\nPackage generated: {package['generated_at']}")
print("All snapshots are tamper-evident (SHA-256 hashed). Ready for auditor delivery.")

## Demo 5: LLM-Powered Compliance Gap Report Generator

**The final stage: turning raw findings into an auditor-ready gap report.**

A compliance gap report needs to:
1. Map each finding to specific control clauses across frameworks
2. Assess severity and risk exposure
3. Recommend remediation with priority order
4. Provide a cross-framework compliance matrix (one finding, multiple framework credits)

This is the task LLMs are exceptionally good at — synthesizing multiple findings
into a coherent narrative that a compliance officer or auditor can act on.

**Cross-framework mapping** is where the real efficiency gain lives:

| One Finding | PCI-DSS | SOC 2 TSC | NIST CSF |
|------------|---------|-----------|----------|
| Telnet enabled | Req 2.2.7 | CC6.1 | PR.DS-2 |
| No remote logging | Req 10.2.1, 10.7 | CC7.2 | DE.CM-1 |
| Orphaned account | Req 8.2.6 | CC6.3 | PR.AC-1 |
| Finance user in FW admins | Req 7.2.2 | CC6.3 | PR.AC-4 |

*Network analogy: cross-framework mapping is dual-stack networking.
One physical interface, two protocol stacks. One control implementation,
two framework credits. Build the network once; run IPv4 and IPv6 simultaneously.*

In [ ]:
# ── Demo 5: LLM-Powered Compliance Gap Report Generator ───────────────────────

@dataclass
class GapFinding:
    finding_id:   str
    category:     str      # config | access | change | evidence
    device:       str
    description:  str
    severity:     str
    affected_frameworks: Dict[str, List[str]]  # {"PCI-DSS": ["Req 2.2.7"], ...}
    raw_evidence: str


# ── Cross-framework mapping table ─────────────────────────────────────────────
# This is the "Rosetta Stone" — one control implementation, multiple framework credits
FRAMEWORK_MAP = {
    "PCI-DSS-2.2.7":   {"PCI-DSS": ["Req 2.2.7"],      "SOC2": ["CC6.1"],         "NIST": ["PR.DS-2", "PR.AC-7"]},
    "PCI-DSS-8.3.6":   {"PCI-DSS": ["Req 8.3.6"],      "SOC2": ["CC6.1"],         "NIST": ["PR.AC-7"]},
    "PCI-DSS-10.2.1":  {"PCI-DSS": ["Req 10.2", "10.7"],"SOC2": ["CC7.2"],        "NIST": ["DE.CM-1"]},
    "PCI-DSS-8.2.6":   {"PCI-DSS": ["Req 8.2.6"],      "SOC2": ["CC6.3"],         "NIST": ["PR.AC-1"]},
    "PCI-DSS-7.2.2":   {"PCI-DSS": ["Req 7.2.2"],      "SOC2": ["CC6.3"],         "NIST": ["PR.AC-4"]},
    "PCI-DSS-1.3.1":   {"PCI-DSS": ["Req 1.3.1", "1.3.2"],"SOC2": ["CC6.6"],     "NIST": ["PR.AC-5"]},
}


def build_gap_findings(
    device_checks: Dict[str, List[ComplianceCheck]],
    access_violations: List[AccessViolation],
    change_impacts: List[ComplianceImpact]
) -> List[GapFinding]:
    """Aggregate all compliance findings into a unified gap list."""
    findings: List[GapFinding] = []
    counter = 0

    # From device config checks
    for device, checks in device_checks.items():
        for c in checks:
            if c.result in ("FAIL", "NEEDS_REVIEW"):
                counter += 1
                fw_map = FRAMEWORK_MAP.get(c.control_id, {"Other": [c.control_id]})
                findings.append(GapFinding(
                    finding_id=f"GAP-{counter:03d}",
                    category="config",
                    device=device,
                    description=c.requirement,
                    severity=c.severity,
                    affected_frameworks=fw_map,
                    raw_evidence=c.evidence
                ))

    # From access violations
    for v in access_violations:
        counter += 1
        fw_controls = defaultdict(list)
        for ctrl in v.controls:
            for fw_name, clauses in FRAMEWORK_MAP.get(ctrl, {"Other": [ctrl]}).items():
                fw_controls[fw_name].extend(clauses)
        findings.append(GapFinding(
            finding_id=f"GAP-{counter:03d}",
            category="access",
            device=f"identity/{v.username}",
            description=v.violation + ": " + v.detail,
            severity=v.severity,
            affected_frameworks=dict(fw_controls),
            raw_evidence=f"Account: {v.username}, Controls: {v.controls}"
        ))

    # From high-impact change requests
    for imp in change_impacts:
        if imp.impact_level in ("CRITICAL", "HIGH"):
            counter += 1
            fw_controls = defaultdict(list)
            for ctrl in imp.controls_at_risk:
                for fw_name, clauses in FRAMEWORK_MAP.get(ctrl, {"Other": [ctrl]}).items():
                    fw_controls[fw_name].extend(clauses)
            findings.append(GapFinding(
                finding_id=f"GAP-{counter:03d}",
                category="change",
                device=f"change/{imp.change_id}",
                description=f"High-impact change: {imp.change_id} (impact={imp.impact_level})",
                severity="critical" if imp.impact_level == "CRITICAL" else "high",
                affected_frameworks=dict(fw_controls),
                raw_evidence=f"Findings: {imp.rule_findings}"
            ))

    return sorted(findings, key=lambda f: ["critical","high","medium","low"].index(f.severity))


def generate_gap_report(findings: List[GapFinding]) -> str:
    """Use LLM Sonnet to generate a structured compliance gap report."""
    if not findings:
        return "No compliance gaps identified."

    finding_text = "\n".join(
        f"{f.finding_id} [{f.severity.upper()}] ({f.category}) {f.device}: "
        f"{f.description} | Frameworks: {f.affected_frameworks}"
        for f in findings[:8]  # top 8 findings for LLM context window
    )

    prompt = (
        "You are a senior compliance auditor writing a gap report for the CISO.\n"
        "Format the following findings into a professional compliance gap report with:\n"
        "1. Executive summary (2 sentences)\n"
        "2. Critical findings (bullet list with framework references)\n"
        "3. Prioritized remediation roadmap (ordered by risk)\n"
        "4. Cross-framework efficiency note (which remediations close multiple framework gaps)\n\n"
        f"FINDINGS:\n{finding_text}\n\n"
        "Keep total response under 300 words. Use markdown formatting."
    )

    if USE_API:
        resp = client.messages.create(
            model="claude-sonnet-4-5-20250514",
            max_tokens=450,
            messages=[{"role": "user", "content": prompt}]
        )
        return resp.content[0].text.strip()
    # Simulation
    return (
        "**COMPLIANCE GAP REPORT — SIMULATION MODE**\n\n"
        "**Executive Summary:** Multiple critical compliance gaps identified across network "
        "device configurations and identity access management. Immediate remediation required "
        "for PCI-DSS scope.\n\n"
        "**Critical Findings:**\n"
        "- Telnet enabled on edge-sw-02 (PCI-DSS 2.2.7, SOC2 CC6.1)\n"
        "- Orphaned accounts (PCI-DSS 8.2.6, SOC2 CC6.3)\n"
        "- Finance user in firewall admin group (PCI-DSS 7.2.2)\n\n"
        "**Remediation Priority:**\n"
        "1. Disable telnet immediately — closes PCI-DSS 2.2.7 + SOC2 CC6.1\n"
        "2. Disable orphaned accounts — closes PCI-DSS 8.2.6 + SOC2 CC6.3\n"
        "3. Remove finance user from Firewall_Admins\n\n"
        "**Cross-framework efficiency:** Fixing access violations simultaneously closes "
        "PCI-DSS Req 7+8, SOC2 CC6.3, and NIST PR.AC-1 findings."
    )


# ── Aggregate all findings and generate report ─────────────────────────────────
print("=" * 65)
print("COMPLIANCE GAP REPORT GENERATOR")
print("=" * 65)

gap_findings = build_gap_findings(
    all_device_results,
    all_violations,
    impacts
)

print(f"\nTotal gap findings: {len(gap_findings)}")
print(f"  Critical: {sum(1 for f in gap_findings if f.severity == 'critical')}")
print(f"  High    : {sum(1 for f in gap_findings if f.severity == 'high')}")

print("\nFramework coverage:")
framework_counts: Dict[str, int] = defaultdict(int)
for f in gap_findings:
    for fw in f.affected_frameworks:
        framework_counts[fw] += 1
for fw, count in sorted(framework_counts.items()):
    print(f"  {fw:12}: {count} finding(s)")

print("\n" + "=" * 65)
print("GENERATING CISO-READY GAP REPORT (claude-sonnet-4-5)...")
print("=" * 65 + "\n")

report = generate_gap_report(gap_findings)
print(report)

print("\n" + "=" * 65)
print("[GUARDRAIL] This report requires review by a qualified compliance officer.")
print("[GUARDRAIL] LLM framework mappings must be validated before audit submission.")
print("[GUARDRAIL] No automated remediation executed. All actions require approval.")

## Summary: What You Built

You now have a working **Compliance Automation** system with 5 engines:

| Engine | Technique | Input | Output |
|--------|-----------|-------|--------|
| **Config Checker** | Deterministic rules + LLM semantic | Device config | ComplianceCheck list |
| **Access Auditor** | Policy engine + HR roster diff | User accounts | AccessViolation list |
| **Change Scorer** | Rule-based impact scoring | Config diff | ComplianceImpact + LLM assessment |
| **Evidence Collector** | Continuous snapshots + SHA-256 | Live configs | Tamper-evident EvidenceStore |
| **Gap Reporter** | LLM Sonnet + framework mapping | All findings | CISO-ready gap report |

### Why Continuous > Point-in-Time

- **Point-in-time**: you are compliant the day of the audit. Unknown state for 363 other days.
- **Continuous**: you know your compliance posture every hour. Drift detected immediately.
- **SOC 2 Type II** explicitly requires continuous evidence — continuous automation is not optional.

### The Cross-Framework Efficiency Principle

```
One control implemented correctly:
  Disable telnet, enforce SSH v2
    -> PCI-DSS Req 2.2.7  PASS
    -> SOC 2 CC6.1        PASS
    -> NIST PR.DS-2       PASS

One fix, three framework credits. This is dual-stack compliance.
```

### Production Upgrade Path

```
Regex config parsing          ->  NAPALM / Netmiko config collection + TextFSM templates
Simulated user inventory      ->  Active Directory via ldap3 or Azure AD Graph API
In-memory evidence store      ->  AWS S3 Object Lock (WORM) or Azure Blob immutable storage
Manual change intake          ->  ServiceNow / Jira change management API integration
claude-haiku-4-5 checks       ->  claude-sonnet-4-5 for complex semantic assessments
```

### The Non-Negotiable Rule

> **Every compliance finding and LLM-generated framework mapping requires
> review by a qualified compliance officer before audit submission.
> Regulatory interpretation requires human judgment.
> The AI accelerates the work; the compliance team owns the result.**

In [ ]:
# ── Chapter 68: Production Deployment Checklist ────────────────────────────────
print("CHAPTER 68: PRODUCTION DEPLOYMENT CHECKLIST")
print("=" * 60)

checklist = [
    # Data collection
    ("Data Collection",   "Config pull: NAPALM get_config() on all in-scope devices"),
    ("Data Collection",   "IAM pull: ldap3 or Azure AD Graph API for user/group data"),
    ("Data Collection",   "Change feed: ServiceNow / ITSM API for approved changes"),
    ("Data Collection",   "Frequency: hourly config snapshots, daily IAM pulls"),
    # Compliance rules
    ("Rules",             "Maintain rule library: one rule file per compliance framework"),
    ("Rules",             "Separate deterministic (regex) from semantic (LLM) checks"),
    ("Rules",             "Version control your rule library — changes need audit trail"),
    ("Rules",             "Framework mapping table: update on every major revision (PCI 4.0, etc.)"),
    # Evidence management
    ("Evidence",          "Immutable storage: S3 Object Lock, Azure Blob WORM, or equiv."),
    ("Evidence",          "Hash every snapshot — SHA-256 at minimum"),
    ("Evidence",          "Retention: PCI-DSS = 12 months, HIPAA = 6 years minimum"),
    ("Evidence",          "Evidence query API: auditors need self-service access"),
    # LLM integration
    ("LLM",               "claude-haiku-4-5-20251001: per-control triage (high volume)"),
    ("LLM",               "claude-sonnet-4-5-20250514: gap reports, semantic assessments"),
    ("LLM",               "Always include config text in prompt — no hallucinated evidence"),
    ("LLM",               "Human review required before using LLM output in audit submission"),
    # Guardrails
    ("Guardrails",        "No automated remediation without explicit change management approval"),
    ("Guardrails",        "Compliance findings escalation: critical = 24h, high = 72h SLA"),
    ("Guardrails",        "Audit trail: log every automated check + who reviewed the result"),
    ("Guardrails",        "Qualified compliance officer must sign off on all gap reports"),
]

current_cat = ""
for cat, item in checklist:
    if cat != current_cat:
        print(f"\n[{cat}]")
        current_cat = cat
    print(f"  + {item}")

print("\n" + "=" * 60)
print("KEY FORMULAS")
print("=" * 60)
print("Compliance drift rate     = non_compliant_snapshots / total_snapshots  (target: < 2%)")
print("Evidence coverage         = controls_with_continuous_evidence / total_controls")
print("Cross-framework leverage  = framework_credits / distinct_controls_implemented")
print("Mean time to detect drift = avg(drift_detected_timestamp - last_compliant_snapshot)")
print("")
print("Dual-stack compliance: one control implemented -> credit in PCI-DSS + SOC2 + NIST.")
print("Build the control once. Map it to all applicable frameworks. Collect evidence once.")
print("\nChapter 68 complete. Automate the evidence. Eliminate the audit scramble.")